# Simple CNN testing

In [1]:
import torch
import wandb
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [2]:
# Check for GPU
device = None
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else: 
    device = torch.device("cpu")

print(device)

cuda


In [3]:
transformer = transforms.Compose([
    transforms.Resize((224, 224)), # transform to size = 224x224
    transforms.ToTensor(), # transforms into tensor
])

full_dataset = datasets.ImageFolder("../data/icosimal_img_class_03/train", transform=transformer)
splits = torch.load("../data/split/split_train_test_indices.pth")

train_dataset = torch.utils.data.Subset(full_dataset, splits['train_idx'])
test_dataset = torch.utils.data.Subset(full_dataset, splits['test_idx'])
val_dataset = datasets.ImageFolder("../data/icosimal_img_class_03/validate", transform=transformer)

In [4]:
# check dataset loaded correctly
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Training dataset size: 20000
Validation dataset size: 6000
Test dataset size: 4000


### Create Training loop for models

In [5]:
def train_eval(model, optimizer, nepochs, batch_size, training_data, validation_data, device, entity='MSE_DeLearn_SPR26', project='MPW-CNN',run_name=None, use_wandb=True):
    """
    Train and evaluate a model.
    Logs train/validation loss and accuracy to Weights & Biases if use_wandb=True.
    """
    cost_hist = []
    cost_hist_test = []
    acc_hist = []
    acc_hist_test = []

    model = model.to(device) # <-- move model to device (GPU or CPU)
    cost_ce = torch.nn.CrossEntropyLoss().to(device)
    
    train_loader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(validation_data, batch_size=batch_size, shuffle=True)
    
    if use_wandb:
        wandb.init(
            entity=entity,
            project=project,
            name=run_name,
            settings=wandb.Settings(init_timeout=300),
            config={
                "epochs": nepochs,
                "batch_size": batch_size,
                "optimizer": optimizer.__class__.__name__,
                "loss": "CrossEntropyLoss",
                "device": str(device),
                "model": model.__class__.__name__
            }
        )
        wandb.watch(model, log="all", log_freq=100)

    for epoch in range(nepochs):
        model.train()
        size = len(train_loader.dataset)
        nbatches = len(train_loader)
        cost, acc = 0.0, 0.0
        for batch, (X, Y) in enumerate(train_loader):
            X,Y = X.to(device),Y.to(device)
            pred = model(X)
            loss = cost_ce(pred, Y)
            cost += loss.item()
            acc += (pred.argmax(dim=1) == Y).type(torch.float).sum().item()

            # gradient, parameter update
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        cost /= nbatches
        acc /= size

        model.eval()
        size_test = len(val_loader.dataset)
        nbatches_test = len(val_loader)
        cost_test, acc_test = 0.0, 0.0     

        with torch.no_grad():
            for X, Y in val_loader:
                X,Y = X.to(device),Y.to(device)
                pred = model(X)
                cost_test += cost_ce(pred, Y).item()
                acc_test += (pred.argmax(dim=1) == Y).type(torch.float).sum().item()

        cost_test /= nbatches_test
        acc_test /= size_test

        print("Epoch %i: %f, %f, %f, %f"%(epoch, cost, acc, cost_test, acc_test))

        cost_hist.append(cost)
        cost_hist_test.append(cost_test)
        acc_hist.append(acc)
        acc_hist_test.append(acc_test)

        if use_wandb:
            wandb.log({
                "epoch": epoch + 1,
                "train_loss": cost,
                "train_accuracy": acc,
                "val_loss": cost_test,
                "val_accuracy": acc_test,
                "lr": optimizer.param_groups[0]['lr']
            })

    if use_wandb:
        wandb.finish()

    return cost_hist, cost_hist_test, acc_hist, acc_hist_test

### Creating shallow CNN-model

In [6]:
# creata simple model with one convolutional layer and two fully connected layers

class simple_model(nn.Module):
    
    def __init__(self, units=100):
        super(simple_model, self).__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # Conv with 32 filters, kernel size 3x3, padding 1
            nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Flatten(),
            nn.Linear(112*112*32,units),
            nn.ReLU(),
            nn.Linear(units,10) # output layer with 10 units for 10 classes
        )
        
    
    def forward(self, x):
        return self.seq(x)

In [10]:
# create an model and its summary

model = simple_model(100) # no need for to(device), it breaks when running on Apple mps chip
from torchsummary import summary
summary(model, (3,224,224),device='cpu')

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 224, 224]             896
              ReLU-2         [-1, 32, 224, 224]               0
         MaxPool2d-3         [-1, 32, 112, 112]               0
           Flatten-4               [-1, 401408]               0
            Linear-5                  [-1, 100]      40,140,900
              ReLU-6                  [-1, 100]               0
            Linear-7                   [-1, 10]           1,010
Total params: 40,142,806
Trainable params: 40,142,806
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.57
Forward/backward pass size (MB): 30.63
Params size (MB): 153.13
Estimated Total Size (MB): 184.33
----------------------------------------------------------------


Initiate Training

In [11]:
batch_size = 32
nepochs = 10
lr = 0.1
units = 100

model = simple_model(units)
optimizer = torch.optim.SGD(params=model.parameters(), lr = lr)
cost_train_sgd, cost_valid_sgd, acc_train_sgd, acc_valid_sgd = train_eval(model, optimizer, nepochs, batch_size, train_dataset, val_dataset, device, entity='MSE_DeLearn_SPR26', project='MPW-CNN', run_name='simple_model_Test_wandb', use_wandb=True)


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: lucas-j-keller98 (MSE_DeLearn_SPR26). Use `wandb login --relogin` to force relogin
wandb: WARNING Unable to render Widget, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core


Epoch 0: 2.091170, 0.254600, 1.803423, 0.358333
Epoch 1: 1.714939, 0.397350, 1.800566, 0.362333
Epoch 2: 1.431196, 0.501250, 1.890748, 0.359167
Epoch 3: 1.034498, 0.646900, 2.179717, 0.355500
Epoch 4: 0.650366, 0.784200, 3.105784, 0.325000
Epoch 5: 0.387829, 0.876450, 3.454881, 0.331833
Epoch 6: 0.290788, 0.911950, 3.914341, 0.326333
Epoch 7: 0.181654, 0.948550, 3.975998, 0.327833
Epoch 8: 0.125500, 0.964100, 4.895840, 0.320333
Epoch 9: 0.165971, 0.955100, 4.959427, 0.326500


wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core


This first model is heavily in a overfitting regime.\
Really bad vallidation accuracy and loss.\
The Result was kind of expected looking at the amount of parameters that are estimted during training (+40 Mio).\
Therefore we tried to construct a shallow model that performs better.
Possibilities to improve the model:
- heavier downsampling before passing into a fully connected layer
    --> adding more layers before fully connectde layers (Conv2d --> ReLu --> MaxPool2d)
- reduce/ make the dense layer smaller (less units)
- add regularization:
    - Dropout
    - optimizer
    - eraly stopping
    - etc.


In [12]:
class shallow_model(nn.Module):
    
    def __init__(self, units=100):
        super(shallow_model, self).__init__()
        self.seq = nn.Sequential(
            # Layer 1---------------------------------------------------------------------------------------
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # Conv with 32 filters, kernel size 3x3, padding 1
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 224x224 -> 112x112

            # Layer 2---------------------------------------------------------------------------------------
            nn.Conv2d(32, 64, kernel_size=3, padding=1), # Conv with 64 filters, kernel size 3x3, padding 1
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 112x112 -> 56x56

            # output layer----------------------------------------------------------------------------------
            nn.Flatten(),
            nn.Linear(56*56*64,units),
            nn.ReLU(),
            nn.Linear(units,10) # output layer with 10 units for 10 classes
        )
        
    
    def forward(self, x):
        return self.seq(x)

In [14]:
# create an model and its summary

model = simple_model(100) # no need for to(device), it breaks when running on Apple mps chip
from torchsummary import summary
summary(model, (3,224,224),device='cpu')

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 224, 224]             896
              ReLU-2         [-1, 32, 224, 224]               0
         MaxPool2d-3         [-1, 32, 112, 112]               0
           Flatten-4               [-1, 401408]               0
            Linear-5                  [-1, 100]      40,140,900
              ReLU-6                  [-1, 100]               0
            Linear-7                   [-1, 10]           1,010
Total params: 40,142,806
Trainable params: 40,142,806
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.57
Forward/backward pass size (MB): 30.63
Params size (MB): 153.13
Estimated Total Size (MB): 184.33
----------------------------------------------------------------


In [13]:
batch_size = 32
nepochs = 10
lr = 0.1
units = 100

model = simple_model(units)
optimizer = torch.optim.SGD(params=model.parameters(), lr = lr)
cost_train_sgd, cost_valid_sgd, acc_train_sgd, acc_valid_sgd = train_eval(model, optimizer, nepochs, batch_size, train_dataset, val_dataset, device, entity='MSE_DeLearn_SPR26', project='MPW-CNN', run_name='shallow_model', use_wandb=True)


wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core


Epoch 0: 2.255288, 0.186200, 2.141300, 0.241333
Epoch 1: 1.982461, 0.296550, 1.851362, 0.343833
Epoch 2: 1.738407, 0.388850, 1.803388, 0.367667
Epoch 3: 1.452711, 0.494950, 1.868406, 0.370333
Epoch 4: 1.101732, 0.622550, 2.119803, 0.351167
Epoch 5: 0.744286, 0.750350, 2.720517, 0.335833
Epoch 6: 0.539178, 0.829250, 3.146812, 0.337000
Epoch 7: 0.377976, 0.879450, 3.877148, 0.324333
Epoch 8: 0.288310, 0.910450, 4.103664, 0.315667
Epoch 9: 0.241948, 0.929350, 4.661720, 0.328833


wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core


## Hyperparameter tuning

WandB has a integrated hyperparameter sweep which we want to try out for this MPW.\
in the following section we tried to tune the model with this function